## Step 0: Environment Setup & Model Initialization
Configuring the hardware device (GPU/CPU) and loading the target Large Language Model (`Qwen/Qwen2.5-Coder-1.5B-Instruct`) alongside its tokenizer.

In [ ]:

#!pip install transformers datasets torch accelerate


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float32
).to(device)

model.eval()
print(f"Model loaded successfully!")

Using device: cuda
Loading Qwen/Qwen2.5-Coder-1.5B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
# --- Persistence setup: run once per session (idempotent) ---
import os

ARTIFACT_DIR = "/content/drive/MyDrive/mbpp_eval"
try:
    from google.colab import drive
    drive.mount("/content/drive")          # no-op if already mounted
except Exception as e:
    ARTIFACT_DIR = os.path.abspath("./mbpp_eval")   # local fallback, NOT disconnect-safe
    print(f"[warn] Drive not mounted ({e}); using {ARTIFACT_DIR} (won't survive a disconnect).")

os.makedirs(ARTIFACT_DIR, exist_ok=True)

def artifact_path(name):
    return os.path.join(ARTIFACT_DIR, name)

print(f"Artifacts -> {ARTIFACT_DIR}")

Mounted at /content/drive
Artifacts -> /content/drive/MyDrive/mbpp_eval


## Step 1: Data Ingestion & Exploration (MBPP Dataset)
Loading the dataset and analyzing its structure to understand the available features and test cases format.

In [ ]:
# Load the MBPP (Mostly Basic Python Problems) dataset
mbpp = load_dataset("google-research-datasets/mbpp", "full", trust_remote_code=True)
print(f"Number of problems in 'test' split: {len(mbpp['test'])}")

# Inspect dataset splits
print("\nDataset Splits:")
for split in mbpp.keys():
    print(f"  - {split}: {len(mbpp[split])} examples")

# Inspect available features
print("\nFeatures (columns):")
print(mbpp['test'].features)

#Examine the structure of a single problem
print("\nExample Problem (Index 0):")
example = mbpp['test'][0]
for key, value in example.items():
    print(f"{key}: {value}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google-research-datasets/mbpp' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google-research-datasets/mbpp' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

full/train-00000-of-00001.parquet:   0%|          | 0.00/87.2k [00:00<?, ?B/s]

full/test-00000-of-00001.parquet:   0%|          | 0.00/116k [00:00<?, ?B/s]

full/validation-00000-of-00001.parquet:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/374 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/10 [00:00<?, ? examples/s]

Number of problems in 'test' split: 500

Dataset Splits:
  - train: 374 examples
  - test: 500 examples
  - validation: 90 examples
  - prompt: 10 examples

Features (columns):
{'task_id': Value('int32'), 'text': Value('string'), 'code': Value('string'), 'test_list': List(Value('string')), 'test_setup_code': Value('string'), 'challenge_test_list': List(Value('string'))}

Example Problem (Index 0):
task_id: 11
text: Write a python function to remove first and last occurrence of a given character from the string.
code: def remove_Occ(s,ch): 
    for i in range(len(s)): 
        if (s[i] == ch): 
            s = s[0 : i] + s[i + 1:] 
            break
    for i in range(len(s) - 1,-1,-1):  
        if (s[i] == ch): 
            s = s[0 : i] + s[i + 1:] 
            break
    return s 
test_list: ['assert remove_Occ("hello","l") == "heo"', 'assert remove_Occ("abcda","a") == "bcd"', 'assert remove_Occ("PHP","P") == "H"']
test_setup_code: 
challenge_test_list: ['assert remove_Occ("hellolloll

## Step 2: Test Case Preparation (Few-Shot Setup)
Preparing the data for few-shot prompting. We split the available test assertions into examples (to provide context to the LLM) and a hidden evaluation test (to measure actual performance).

In [ ]:
def split_tests(problem):
    """
    Splits the test_list into few-shot examples and a hidden evaluation test.

    Args:
        problem (dict): A problem dictionary from the MBPP dataset.

    Returns:
        tuple: (example_tests, eval_test, valid_split)
            - example_tests (list): First 2 tests used for few-shot prompting.
            - eval_test (list): The 3rd test wrapped in a list for evaluation.
            - valid_split (bool): True if the problem has at least 3 tests.
    """
    test_list = problem.get('test_list', [])

    # Filter out problems with insufficient test cases
    if len(test_list) < 3:
        return [], [], False

    # Extract tests for prompt context and evaluation
    example_tests = test_list[:2]
    eval_test = [test_list[2]] # Wrapped in a list for the execution engine

    return example_tests, eval_test, True

## Step 3: Code Generation & Execution Engine
Building the core pipeline components:
1. **Execution Engine (`test_code`)**: Safely executes the LLM-generated Python code and verifies it against the hidden test cases.
2. **Generation Function (`generate_code`)**: Constructs the prompt, handles tokenization via the chat template, and parses the raw output to extract clean Python code.

In [ ]:
def test_code(generated_code, test_cases):
    """
    Dynamically executes generated code and runs it against test cases.

    Returns:
        tuple: (code_compiled: bool, tests_passed: bool)
    """
    namespace = {}

    # Step 1: Check if the code compiles and runs without syntax errors
    try:
        exec(generated_code, namespace)
    except Exception:
        return False, False

    if not test_cases:
        return True, False

    # Step 2: Run the hidden assertions
    try:
        for test in test_cases:
            exec(test, namespace)
        return True, True
    except Exception:
        return True, False


def generate_code(problem_text, example_tests, max_new_tokens=512):
    """
    Generates Python code for a given problem using the loaded LLM.
    Extracts only the code block from the model's response.
    """
    tests_str = "\n".join(example_tests)

    # Baseline Prompt
    prompt = f"""Write a Python function to solve this problem:
{problem_text}

The function should pass these test cases:
{tests_str}

Write only the function code, inside a ```python code block.
"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Parsing the output to extract code
    if "```python" in response:
        code = response.split("```python")[1].split("```")[0].strip()
    elif "```" in response:
        code = response.split("```")[1].split("```")[0].strip()
    else:
        code = response.strip()

    return code

# ---------------------------------------------------------
# Sanity Check: Test the pipeline on a small sample
# ---------------------------------------------------------
print("Sanity Check: Running generation and execution on 3 sample problems...\n")
selected_indices = [5, 15, 30]

for idx in selected_indices:
    problem = mbpp['test'][idx]
    example_tests, eval_test, valid = split_tests(problem)

    if not valid:
        continue

    print(f"{'='*60}")
    print(f"Task ID {idx}: {problem['text']}")

    code = generate_code(problem['text'], example_tests)
    print(f"\nGenerated Code:\n{code}\n")

    compiled, passed = test_code(code, eval_test)

    print("Execution Analysis:")
    if passed:
        print("✅ Success: Code compiles and passes all hidden tests.")
    elif compiled:
        print("⚠️ Logic Error: Code compiles but fails the hidden test assertions.")
    else:
        print("❌ Syntax Error: Code fails to compile.")
    print("-" * 60)

Sanity Check: Running generation and execution on 3 sample problems...

Task ID 5: Write a function to find sequences of lowercase letters joined with an underscore.

Generated Code:
import re

def text_lowercase_underscore(text):
    # Define the pattern for lowercase letters followed by an underscore
    pattern = '^[a-z]+_[a-z]+$'
    
    # Search for the pattern in the given text
    if re.search(pattern, text):
        return 'Found a match!'
    else:
        return 'Not matched!'

Execution Analysis:
✅ Success: Code compiles and passes all hidden tests.
------------------------------------------------------------
Task ID 15: Write a function to check if the given tuple list has all k elements.

Generated Code:
def check_k_elements(lst, k):
    return len(lst) == k

Execution Analysis:
✅ Success: Code compiles and passes all hidden tests.
------------------------------------------------------------
Task ID 30: Write a function to filter even numbers using lambda function.

Gener

### Sanity Check Observations
To verify the execution engine before running the full evaluation, a sanity check was performed on 3 sampled problems.

The model successfully generated correct Python code for all selected examples. The output was syntactically valid, utilized appropriate Python logic (such as regex and list comprehensions), and successfully passed the hidden evaluation tests. This confirms that the base generation and execution pipeline is robust and ready for large-scale evaluation.

## Step 4: Baseline Evaluation
Running the generation and execution pipeline across a sample of 100 MBPP problems to establish a baseline pass rate for the `Qwen2.5-Coder` model using the naive prompt.

In [ ]:
print("Starting baseline evaluation on 100 problems...\n")

n_problems = 100

results = []
passed_count = 0
valid_count = 0

for i in range(n_problems):
    problem = mbpp['test'][i]
    example_tests, eval_test, valid = split_tests(problem)

    if not valid:
        continue

    valid_count += 1

    # Generate code using the baseline prompt
    code = generate_code(problem['text'], example_tests)

    # Evaluate execution and correctness
    compiled, passed = test_code(code, eval_test)

    if passed:
        passed_count += 1

    # Store results for subsequent error analysis
    results.append({
        'task_id': i,
        'text': problem['text'],
        'example_tests': example_tests,
        'eval_test': eval_test,
        'code': code,
        'compiled': compiled,
        'passed': passed
    })

print(f"\nBaseline Pass Rate: {passed_count}/{valid_count} ({passed_count/valid_count:.2%})")

Starting baseline evaluation on 100 problems...


Baseline Pass Rate: 60/100 (60.00%)


## Step 5: Error Analysis & Prompt Engineering
Analyzing the failure modes from the baseline evaluation. Many compilation or logic errors stem from missing imports or ignored edge cases.

To address this, we implement an **Improved Prompt** that enforces strict constraints (explicit imports, edge-case handling, and strict formatting), and measure the delta in the overall pass rate.

In [ ]:
# Calculate baseline metrics
passed_count = sum(1 for r in results if r['passed'])
compiled_count = sum(1 for r in results if r['compiled'])
valid_count = len(results)
failed_examples = [r for r in results if not r['passed']]

print("="*70)
print("BASELINE ERROR ANALYSIS")
print("="*70)

print(f"Final Pass Rate (Baseline): {passed_count}/{valid_count} ({passed_count/valid_count:.2%})")
print(f"Compilation Rate: {compiled_count}/{valid_count} ({compiled_count/valid_count:.2%})")
print(f"Total Failures: {len(failed_examples)}\n")

print("Examining 3 specific failed cases:\n")
for i, fail in enumerate(failed_examples[:3], 1):
    print(f"[FAILED CASE #{i}] Task ID: {fail['task_id']}")
    print(f"Prompt: {fail['text']}")
    print(f"Code:\n{fail['code']}")
    print("-" * 50)


def generate_code_improved(problem_text, example_tests, max_new_tokens=512):
    """
    Generates Python code using an engineered prompt with strict constraints.
    """
    tests_str = "\n".join(example_tests)

    prompt = f"""You are an expert Python programmer.
Task: Write a Python function to solve this problem:
{problem_text}

Constraints:
1. The function MUST pass these specific tests:
{tests_str}
2. Import ALL necessary libraries (math, re, collections, etc) explicitly.
3. Handle edge cases.
4. Return the result (do NOT print).

Write ONLY the code inside a ```python block.
"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    if "```python" in response:
        return response.split("```python")[1].split("```")[0].strip()
    elif "```" in response:
        return response.split("```")[1].split("```")[0].strip()
    return response.strip()

# ---------------------------------------------------------
# Step 5A: Retry specific failures with the improved prompt
# ---------------------------------------------------------
print("\n" + "="*70)
print("PROMPT ENGINEERING: Retrying Failed Cases")
print("="*70)

for i, fail in enumerate(failed_examples[:3], 1):
    print(f"\nRetry Case #{i} (Task {fail['task_id']})...")
    new_code = generate_code_improved(fail['text'], fail['example_tests'])
    print(f"[NEW CODE]:\n{new_code}\n")

    _, passed = test_code(new_code, fail['eval_test'])
    print(f"Result: {'✅ FIXED!' if passed else '❌ STILL FAILED'}")

# ---------------------------------------------------------
# Step 5B: Rerun evaluation on the entire dataset
# ---------------------------------------------------------
print("\n" + "="*70)
print("FULL PIPELINE EVALUATION: Improved Prompt")
print("="*70)

improved_passed_count = 0

for i in range(valid_count):
    task_id = results[i]['task_id']
    problem_text = results[i]['text']
    example_tests = results[i]['example_tests']
    eval_test = results[i]['eval_test']

    try:
        code = generate_code_improved(problem_text, example_tests)
        _, passed = test_code(code, eval_test)
        if passed:
            improved_passed_count += 1
    except Exception:
        pass

# ---------------------------------------------------------
# Final Conclusion & Delta
# ---------------------------------------------------------
print("\n" + "="*70)
print("FINAL EVALUATION DELTA")
print("="*70)
print(f"Baseline Pass Rate: {passed_count}/{valid_count} ({passed_count/valid_count:.2%})")
print(f"Improved Pass Rate: {improved_passed_count}/{valid_count} ({improved_passed_count/valid_count:.2%})")

diff = improved_passed_count - passed_count
if diff > 0:
    print(f"\nConclusion: The pass rate IMPROVED by {diff} problems through structured prompt engineering!")
elif diff < 0:
    print(f"\nConclusion: The pass rate DECLINED by {abs(diff)} problems.")
else:
    print("\nConclusion: The pass rate remained the SAME.")



BASELINE ERROR ANALYSIS
Final Pass Rate (Baseline): 60/100 (60.00%)
Compilation Rate: 100/100 (100.00%)
Total Failures: 40

Examining 3 specific failed cases:

[FAILED CASE #1] Task ID: 2
Prompt: Write a function to count the most common words in a dictionary.
Code:
from collections import Counter

def count_common(words):
    # Count the occurrences of each word using Counter
    word_counts = Counter(words)
    # Find the maximum occurrence count
    max_count = max(word_counts.values())
    # Filter words that have the maximum occurrence count
    common_words = [(word, count) for word, count in word_counts.items() if count == max_count]
    return common_words
--------------------------------------------------
[FAILED CASE #2] Task ID: 10
Prompt: Write a function to find m number of multiples of n.
Code:
def multiples_of_num(n, m):
    return [n * i for i in range(1, m + 1)]
--------------------------------------------------
[FAILED CASE #3] Task ID: 13
Prompt: Write a function to 

## Step 6: Failure Analysis — Automated Error Categorization

The prompt-engineering experiment above moved Pass@1 from 60% to 61% — a single problem. That motivates a sharper question: *how* do the remaining 40 failures actually fail?

This step replaces the binary `(compiled, passed)` signal with a concrete failure type per problem. Because these 40 snippets already executed once (safely) during the baseline, a simple in-process `try/except` is sufficient here; the process-isolated, timeout-protected harness (Step 8) is reserved for the untested, higher-variance generations in Step 9. The output, `error_analysis_results.json`, is the single input to the Recovery Analysis.

In [ ]:
# === B1a: capture HOW each baseline failure fails ===
# These 40 snippets already ran safely during the baseline, so an in-process
# try/except is enough here (the subprocess+timeout harness comes later, for B3).
import json
from collections import Counter

failed_examples = [r for r in results if not r["passed"]]
print(f"Analyzing {len(failed_examples)} baseline failures...\n")

def capture_error_type(code, eval_test):
    ns = {}
    try:
        exec(code, ns)
    except Exception as e:
        return type(e).__name__            # failed at definition (e.g. SyntaxError)
    try:
        for test in eval_test:
            exec(test, ns)
    except Exception as e:
        return type(e).__name__            # failed the hidden assertion (e.g. AssertionError)
    return "NoError"                        # guard: a "failure" that unexpectedly re-passes

error_analysis_results = []
for r in failed_examples:
    error_type = capture_error_type(r["code"], r["eval_test"])
    error_analysis_results.append({
        "task_id":       r["task_id"],
        "text":          r["text"],
        "example_tests": r["example_tests"],
        "eval_test":     r["eval_test"],
        "code":          r["code"],
        "error_type":    error_type,
    })

out_path = artifact_path("error_analysis_results.json")
with open(out_path, "w") as f:
    json.dump(error_analysis_results, f, indent=2)
print(f"Saved {len(error_analysis_results)} records -> {out_path}\n")

counts = Counter(r["error_type"] for r in error_analysis_results)
width = max(len("Failure Type"), max(len(k) for k in counts))
print(f"{'Failure Type'.ljust(width)} | Count")
print(f"{'-' * width}-+------")
for error_type, n in counts.most_common():
    print(f"{error_type.ljust(width)} | {n}")

n_assert = counts.get("AssertionError", 0)
gate = "RUN B1b" if 25 <= n_assert <= 30 else "SKIP B1b"
print(f"\nAssertionError = {n_assert}/{len(error_analysis_results)}  ->  {gate}")
if counts.get("NoError"):
    print(f"[note] {counts['NoError']} snippet(s) re-passed on replay — check those task_ids.")

Analyzing 40 baseline failures...

Saved 40 records -> /content/drive/MyDrive/mbpp_eval/error_analysis_results.json

Failure Type   | Count
---------------+------
AssertionError | 35
TypeError      | 2
IndexError     | 1
NameError      | 1
KeyError       | 1

AssertionError = 35/40  ->  SKIP B1b


In [ ]:
# === B1b: enrich AssertionError records with actual vs expected values ===
# Bare asserts raise an empty message, so we recompute both sides from the test string.
import ast, json

def _contains_call(node):
    return any(isinstance(n, ast.Call) for n in ast.walk(node))

def parse_expected_actual(code, test_str):
    """Return (actual_repr, expected_repr, note); reprs are None when not extractable."""
    ns = {}
    try:
        exec(code, ns)
    except Exception as e:
        return None, None, f"exec failed: {type(e).__name__}"
    try:
        tree = ast.parse(test_str.strip())
    except SyntaxError as e:
        return None, None, f"unparseable test: {e}"
    if not tree.body or not isinstance(tree.body[0], ast.Assert):
        return None, None, "not a single assert"
    cond = tree.body[0].test
    if not (isinstance(cond, ast.Compare) and len(cond.ops) == 1 and isinstance(cond.ops[0], ast.Eq)):
        return None, None, "non-equality assertion"
    left, right = cond.left, cond.comparators[0]
    l_call, r_call = _contains_call(left), _contains_call(right)
    if l_call and not r_call:
        actual_node, expected_node = left, right
    elif r_call and not l_call:
        actual_node, expected_node = right, left
    else:
        actual_node, expected_node = left, right   # fallback: MBPP convention func(...) == expected
    def _eval(node):
        try:
            return repr(eval(ast.unparse(node), ns)), None
        except Exception as e:
            return None, f"eval {type(e).__name__} on {ast.unparse(node)!r}"
    a_repr, a_err = _eval(actual_node)
    e_repr, e_err = _eval(expected_node)
    return a_repr, e_repr, ("; ".join(x for x in (a_err, e_err) if x) or None)

path = artifact_path("error_analysis_results.json")
records = json.load(open(path))

clean, flagged = 0, 0
print(f"{'task':>5} | {'expected':<20} | {'actual':<20} | note")
print("-" * 70)
for r in records:
    if r["error_type"] != "AssertionError":
        continue
    actual, expected, note = parse_expected_actual(r["code"], r["eval_test"][0])
    r["actual_value"], r["expected_value"] = actual, expected
    if note:
        r["parse_note"] = note
        flagged += 1
    else:
        clean += 1
    print(f"{r['task_id']:>5} | {str(expected)[:20]:<20} | {str(actual)[:20]:<20} | {note or ''}")

with open(path, "w") as f:
    json.dump(records, f, indent=2)

n_assert = sum(1 for r in records if r["error_type"] == "AssertionError")
print(f"\nEnriched {n_assert} AssertionError records — {clean} parsed cleanly, {flagged} flagged.")
print(f"Re-saved -> {path}")

 task | expected             | actual               | note
----------------------------------------------------------------------
    2 | [('Apple', 2), ('Ama | [('Apple', 2), ('Ama | 
   10 | [2, 4, 6, 8, 10, 12, | [9, 18]              | 
   16 | ['wonder', 'wonder', | []                   | 
   20 | [6, 5, 7, 8, 1]      | [1, 2, 6, 3, 4]      | 
   23 | 4                    | 0                    | 
   25 | 3                    | 5                    | 
   26 | [1, 10, 12, 19, 'blu | [1, 10, 12, 19, 'red | 
   27 | 10                   | 0.45454545454545453  | 
   28 | 'cdabcd'             | True                 | 
   32 | 'Not matched!'       | 'Found a match!'     | 
   36 | 2                    | 1                    | 
   37 | 31                   | 2863311550           | 
   39 | (2, [1, 2])          | [1, 2]               | 
   45 | True                 | False                | 
   48 | 645                  | 322                  | 
   49 | 1                    | 5             

**Analysis of the Failures:**

As seen in the output above, there are **zero SyntaxErrors**, consistent with the 100% compilation rate — the model writes valid Python but gets the *logic* wrong.

Failures are overwhelmingly **AssertionError (35/40)**: the code runs and returns a value, but the wrong one. By recovering the actual vs. expected values (e.g., a `list` where a `tuple` is expected, inverted booleans, or correct elements in the wrong order), we can see these are real logic/spec gaps, measured relative to the MBPP reference implementation.

## Step 7: Evaluation Infrastructure — Process-Isolated, Timeout-Protected Execution

The Recovery Analysis (Step 9) generates hundreds of new programs with stochastic decoding. These are untested and higher-variance than greedy output, so a single pathological generation (e.g. an infinite loop) could hang the entire run. To contain that, generated code is executed in a **separate, timeout-bounded process** rather than in-process.

This is process isolation plus a timeout — **not** a security sandbox (no containerization). The candidate code is embedded as a `repr()`'d string literal inside a fixed, always-valid wrapper and `exec`'d at runtime inside a `try`, so even a `SyntaxError` is caught like any ordinary exception with no separate return-code path. Each result is emitted on a sentinel-tagged line — making parsing immune to stray stdout from the generated code — and normalized to a uniform `{compiled, passed, error_type, error_message}` schema, including a `Timeout` type and a `NoOutput` guard for hard crashes.

In [ ]:
# === B2: process-isolated, timeout-protected execution harness (for B3) ===
# This is for B3's untested, temperature>0 generations, where a single pathological
# sample (infinite loop) could hang the run. It is process-isolated + timeout-bounded,
# NOT a security sandbox (no Docker). The code is exec'd as a repr'd string inside a
# try, so SyntaxError is caught like any other exception.
import subprocess, sys, json

def run_with_timeout(generated_code, test_cases, timeout=5):
    test_block = "\n".join(test_cases)
    child = (
        "import json, sys\n"
        "def emit(d): print('@@RESULT@@' + json.dumps(d))\n"
        "ns = {}\n"
        "try:\n"
        f"    exec({generated_code!r}, ns)\n"
        "except Exception as e:\n"
        "    emit({'compiled': False, 'passed': False,"
        " 'error_type': type(e).__name__, 'error_message': str(e)[:300]}); sys.exit()\n"
        "try:\n"
        f"    exec({test_block!r}, ns)\n"
        "    emit({'compiled': True, 'passed': True,"
        " 'error_type': None, 'error_message': None})\n"
        "except Exception as e:\n"
        "    emit({'compiled': True, 'passed': False,"
        " 'error_type': type(e).__name__, 'error_message': str(e)[:300]})\n"
    )
    try:
        proc = subprocess.run([sys.executable, "-c", child], timeout=timeout,
                              capture_output=True, text=True, encoding="utf-8")
    except subprocess.TimeoutExpired:
        return {"compiled": True, "passed": False,
                "error_type": "Timeout", "error_message": f">{timeout}s"}
    for line in reversed(proc.stdout.splitlines()):
        if line.startswith("@@RESULT@@"):
            return json.loads(line[len("@@RESULT@@"):])
    return {"compiled": None, "passed": False, "error_type": "NoOutput",
            "error_message": (proc.stderr.strip()[-300:] or f"rc={proc.returncode}")}

# --- self-check (fast; no infinite-loop case to keep it snappy) ---
assert run_with_timeout("def f(x):\n return x*2", ["assert f(3)==6"])["passed"] is True
assert run_with_timeout("def f(x):\n return x*2", ["assert f(3)==7"])["error_type"] == "AssertionError"
assert run_with_timeout("def f(x)\n return x", ["assert f(1)==1"])["error_type"] == "SyntaxError"
print("B2 harness OK.")

B2 harness OK.


## Step 8: Recovery Analysis — Can Stochastic Decoding Recover Greedy Failures?

This is the headline experiment. On the 40 problems that greedy decoding failed, the prompt is held **identical to the baseline** and only the decoding changes: `do_sample=True`, temperature = 0.8, top_p = 0.95, generating 10 samples per problem (400 generations total).

Each sample is scored through the Step 8 process-isolated harness against its held-out test. To ensure stability during this heavy GPU computation, the run checkpoints to Google Drive after every problem, so a Colab disconnect costs at most 10 generations. Let's run the generations and calculate the recovery rates.

In [ ]:
# === B3: Recovery Analysis on Failed Tasks — generation + checkpointing (GPU) ===
import json, os, time, torch

NUM_SAMPLES = 10
TEMPERATURE = 0.8
TOP_P       = 0.95
TIMEOUT     = 5          # seconds, per sample, via the B2 harness

def extract_code(response):
    if "```python" in response:
        return response.split("```python")[1].split("```")[0].strip()
    elif "```" in response:
        return response.split("```")[1].split("```")[0].strip()
    return response.strip()

def generate_samples(problem_text, example_tests, k, max_new_tokens=512):
    """k stochastic samples for one problem. Same baseline prompt as generate_code;
    only the decoding is stochastic. Falls back to per-sample generation on OOM."""
    tests_str = "\n".join(example_tests)
    prompt = f"""Write a Python function to solve this problem:
{problem_text}

The function should pass these test cases:
{tests_str}

Write only the function code, inside a ```python code block.
"""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    prompt_len = model_inputs.input_ids.shape[1]

    def _gen(n):
        with torch.no_grad():
            out = model.generate(**model_inputs, max_new_tokens=max_new_tokens,
                                  do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
                                  num_return_sequences=n,
                                  pad_token_id=tokenizer.eos_token_id)
        return [tokenizer.decode(o[prompt_len:], skip_special_tokens=True) for o in out]

    try:
        responses = _gen(k)                      # batched: all k at once
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        torch.cuda.empty_cache()                 # fallback: one sample at a time
        responses = []
        for _ in range(k):
            responses.extend(_gen(1))
    return [extract_code(r) for r in responses]

# ---- resumable loop: one problem (10 samples) at a time ----
problems  = json.load(open(artifact_path("error_analysis_results.json")))
ckpt_path = artifact_path("recovery_analysis_results.json")
recovery  = json.load(open(ckpt_path)) if os.path.exists(ckpt_path) else {}

def _save():
    with open(ckpt_path, "w") as f:
        json.dump(recovery, f, indent=2)

done = sum(1 for p in problems
           if len(recovery.get(str(p["task_id"]), {}).get("samples", [])) >= NUM_SAMPLES)
print(f"Resuming: {done}/{len(problems)} problems already complete.\n")

for p in problems:
    tid = str(p["task_id"])
    rec = recovery.get(tid, {"task_id": p["task_id"], "error_type": p["error_type"], "samples": []})
    need = NUM_SAMPLES - len(rec["samples"])
    if need <= 0:
        continue
    t0 = time.time()
    for code in generate_samples(p["text"], p["example_tests"], need):
        r = run_with_timeout(code, p["eval_test"], timeout=TIMEOUT)
        rec["samples"].append({"code": code, "passed": bool(r["passed"]),
                               "error_type": r["error_type"]})
    rec["recovered_count"] = sum(1 for s in rec["samples"] if s["passed"])
    rec["recovered"]       = rec["recovered_count"] > 0
    recovery[tid] = rec
    _save()   # checkpoint after each problem -> a disconnect costs <=10 generations
    flag = "RECOVERED" if rec["recovered"] else ""
    print(f"task {p['task_id']:>3} ({p['error_type']:<14}): "
          f"{rec['recovered_count']:>2}/{NUM_SAMPLES} passed  {flag}  [{time.time()-t0:.0f}s]")

print("\nGeneration complete — all 40 problems have 10 samples each.")

Resuming: 0/40 problems already complete.

task   2 (AssertionError):  0/10 passed    [23s]
task  10 (AssertionError):  1/10 passed  RECOVERED  [14s]
task  13 (TypeError     ):  3/10 passed  RECOVERED  [18s]
task  14 (IndexError    ):  6/10 passed  RECOVERED  [14s]
task  16 (AssertionError):  0/10 passed    [9s]
task  20 (AssertionError):  0/10 passed    [33s]
task  22 (NameError     ):  0/10 passed    [23s]
task  23 (AssertionError):  3/10 passed  RECOVERED  [22s]
task  25 (AssertionError):  0/10 passed    [32s]
task  26 (AssertionError):  5/10 passed  RECOVERED  [18s]
task  27 (AssertionError):  0/10 passed    [19s]
task  28 (AssertionError):  0/10 passed    [29s]
task  32 (AssertionError):  4/10 passed  RECOVERED  [15s]
task  34 (TypeError     ):  3/10 passed  RECOVERED  [15s]
task  36 (AssertionError):  3/10 passed  RECOVERED  [26s]
task  37 (AssertionError):  0/10 passed    [18s]
task  39 (AssertionError):  6/10 passed  RECOVERED  [16s]
task  45 (AssertionError):  6/10 passed  REC

In [ ]:
# === B3: Recovery Analysis — reporting (reads the checkpoint; no GPU) ===
import json, math
from collections import defaultdict

NUM_SAMPLES = 10
recovery = json.load(open(artifact_path("recovery_analysis_results.json")))
records  = list(recovery.values())
N = len(records)

# recompute c and n directly from samples (don't trust stored fields)
for r in records:
    r["_n"] = len(r["samples"])
    r["_c"] = sum(1 for s in r["samples"] if s["passed"])
if any(r["_n"] != NUM_SAMPLES for r in records):
    print("[warn] some problems do not have exactly 10 samples — re-run Cell 5.\n")

recovered = sum(1 for r in records if r["_c"] > 0)
print("="*64)
print("RECOVERY ANALYSIS ON FAILED TASKS")
print("="*64)
print(f"Recovery@10 = {recovered}/{N} ({recovered/N:.1%})")
print("(Of the 40 greedy failures, how many had >=1 of 10 sampled generations pass.)\n")

# --- stratify by the B1 failure type ---
strata = defaultdict(lambda: {"n": 0, "rec": 0})
for r in records:
    s = strata[r["error_type"]]; s["n"] += 1; s["rec"] += int(r["_c"] > 0)

print(f"{'Failure Type':<16} | {'Recovered':>9} | {'Rate':>5} | note")
print("-"*60)
for et, s in sorted(strata.items(), key=lambda kv: -kv[1]["n"]):
    note = "" if s["n"] >= 10 else "indicative only (small n)"
    print(f"{et:<16} | {s['rec']:>4}/{s['n']:<4}| {s['rec']/s['n']:>4.0%} | {note}")
pn = sum(s["n"] for et, s in strata.items() if et != "AssertionError")
pr = sum(s["rec"] for et, s in strata.items() if et != "AssertionError")
if pn:
    print(f"{'(non-Assertion)':<16} | {pr:>4}/{pn:<4}| {pr/pn:>4.0%} | pooled tail (all small-n)")

# --- bonus: Recovery@k vs sampling budget (unbiased estimator, same 400 samples) ---
def recovery_at_k(records, k):
    vals = []
    for r in records:
        n, c = r["_n"], r["_c"]
        term = (math.comb(n - c, k) / math.comb(n, k)) if (n - c) >= k else 0.0
        vals.append(1.0 - term)
    return sum(vals) / len(vals)

print("\nRecovery@k vs sampling budget (averaged over 40 tasks, from the same samples):")
for k in (1, 3, 5, 10):
    print(f"  Recovery@{k:<2} = {recovery_at_k(records, k):.1%}")

RECOVERY ANALYSIS ON FAILED TASKS
Recovery@10 = 25/40 (62.5%)
(Of the 40 greedy failures, how many had >=1 of 10 sampled generations pass.)

Failure Type     | Recovered |  Rate | note
------------------------------------------------------------
AssertionError   |   22/35  |  63% | 
TypeError        |    2/2   | 100% | indicative only (small n)
IndexError       |    1/1   | 100% | indicative only (small n)
NameError        |    0/1   |   0% | indicative only (small n)
KeyError         |    0/1   |   0% | indicative only (small n)
(non-Assertion)  |    3/5   |  60% | pooled tail (all small-n)

Recovery@k vs sampling budget (averaged over 40 tasks, from the same samples):
  Recovery@1  = 23.2%
  Recovery@3  = 44.3%
  Recovery@5  = 53.1%
  Recovery@10 = 62.5%


### Analysis of the Recovery Results

**Recovery@10 on failed tasks = 25/40 (62.5%)** Nearly two-thirds of the greedy failures have at least one passing program within 10 samples. As seen in the output above, only `AssertionError` has a usable sample size (63% recovery rate). The four tail categories (n ≤ 2) are indicative only, but suggest that shallow runtime bugs are easily sampled around.

**Recovery vs. Sampling Budget (Recovery@k):**
From the same 400 generations, the unbiased estimator `Recovery@k = 1 − C(n−c, k) / C(n, k)` shows that a single stochastic sample (the same compute budget as greedy) already recovers ~23%, with clear diminishing returns past k = 5 (53.1%).

**Key Takeaways:**
* **Decoding Strategy > Prompt Engineering:** Prompt engineering bought us +1 problem; stochastic decoding on the same prompt recovered 25. The correct programs were largely already in the model's distribution — greedy decoding just was not reaching them.
* **Composite Score:** Combining both phases, **~85/100 problems are solvable under a greedy-then-10-sample budget** (60 greedy passes + 25 recovered). Note: This is explicitly *not* Pass@10 on the full MBPP, as the 60 greedy passes were not re-sampled and the budget is asymmetric.
* **Definition of Success:** "Recovered" means passing the single held-out test — the same yardstick as Pass@1. It is an apples-to-apples comparison, not a stronger correctness claim.